# Módulo 36 · RDKit I: moléculas, formatos y descriptores

Cuaderno del curso *Aprende Python desde cero*. Ejecuta las celdas en orden (**Entorno de ejecución → Ejecutar todo**, o Shift+Enter celda a celda). Las explicaciones completas están en el módulo 36 de la web.

In [ ]:
# Instalación (solo la primera vez en Colab; en tu ordenador: pip install rdkit seaborn pandas matplotlib)
!pip -q install rdkit seaborn


Archivos de datos del curso (se crean en la carpeta de trabajo):

In [ ]:
# Archivos de datos del curso
open("aspirin.mol", "w").write("\n     RDKit          2D\n\n 13 13  0  0  0  0  0  0  0  0999 V2000\n    5.2500   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    3.7500   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    3.0000   -2.5981    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    3.0000    0.0000    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    1.5000    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7500   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.7500   -1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.5000    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.7500    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7500    1.2990    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    1.5000    2.5981    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7500    3.8971    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    3.0000    2.5981    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n  1  2  1  0\n  2  3  2  0\n  2  4  1  0\n  4  5  1  0\n  5  6  2  0\n  6  7  1  0\n  7  8  2  0\n  8  9  1  0\n  9 10  2  0\n 10 11  1  0\n 11 12  2  0\n 11 13  1  0\n 10  5  1  0\nM  END\n")
print("Archivos creados:", ["aspirin.mol"])

> 🧪 **RDKit no se puede ejecutar en el navegador**: no existe una versión de RDKit para Python compilada para la web (ni siquiera en Pyodide). En este módulo el código se muestra para ejecutarlo en **Google Colab** o en tu ordenador. Abre `notebooks/modulo-36-rdkit-1.ipynb` (viene en la carpeta del curso) en [colab.research.google.com](https://colab.research.google.com) con *Archivo → Subir cuaderno*: incluye la instalación, los archivos de datos y la comprobación de cada ejercicio.

**RDKit** es la librería de referencia en quimioinformática. Convierte representaciones de texto de una molécula en un objeto `Mol` con el que puedes dibujar, calcular propiedades, buscar subestructuras o comparar fármacos.

> 🔑 **Formatos químicos:** **SMILES**: una línea de texto (`CCO` = etanol). **SMARTS**: patrones para buscar subestructuras. **MOL/SDF**: tabla de átomos con coordenadas y enlaces (SDF = varios MOL en un archivo). **InChI/InChIKey**: identificadores estándar únicos.

In [ ]:
from rdkit import Chem

etanol = Chem.MolFromSmiles("CCO")
print("Átomos:", etanol.GetNumAtoms(), [a.GetSymbol() for a in etanol.GetAtoms()])

aspirina = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")
print("SMILES canónico:", Chem.MolToSmiles(aspirina))
aspirina    # en Jupyter/Colab, la última línea de la celda se dibuja sola

In [ ]:
print("InChI:", Chem.MolToInchi(aspirina))
print("InChIKey:", Chem.MolToInchiKey(aspirina))
paracetamol = Chem.MolFromInchi("InChI=1S/C8H9NO2/c1-6(10)9-7-2-4-8(11)5-3-7/h2-5,11H,1H3,(H,9,10)")
print("Desde InChI:", Chem.MolToSmiles(paracetamol))

In [ ]:
from rdkit import Chem

desde_mol = Chem.MolFromMolFile("aspirin.mol")        # archivo MOL real del curso
print("Desde MOL:", Chem.MolToSmiles(desde_mol))
print(Chem.MolToMolBlock(desde_mol)[:300])

malo = Chem.MolFromSmiles("C1CC")      # anillo sin cerrar
print("SMILES inválido devuelve:", malo)

> ⚠️ Error habitual: no comprobar si `MolFromSmiles` devolvió `None`. Con un SMILES inválido no hay excepción: obtienes None y el error aparece más tarde, en otra línea. Comprueba siempre `if mol is None`.

### Visualizar varias moléculas

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

farmacos = {"Aspirina": "CC(=O)OC1=CC=CC=C1C(=O)O",
            "Paracetamol": "CC(=O)NC1=CC=C(C=C1)O",
            "Ibuprofeno": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
            "Cafeína": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"}
mols = [Chem.MolFromSmiles(s) for s in farmacos.values()]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(200, 170), legends=list(farmacos))

### Descriptores: propiedades calculadas

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

mol = Chem.MolFromSmiles("CC(=O)NC1=CC=C(C=C1)O")     # paracetamol
print("Masa molecular:", round(Descriptors.MolWt(mol), 2))
print("LogP:", round(Descriptors.MolLogP(mol), 2), "(lipofilia: >0 prefiere grasa, <0 agua)")
print("Átomos pesados:", Descriptors.HeavyAtomCount(mol))
print("Enlaces rotables:", Descriptors.NumRotatableBonds(mol))
print("Donadores / aceptores de H:", Descriptors.NumHDonors(mol), "/", Descriptors.NumHAcceptors(mol))
print("TPSA:", Descriptors.TPSA(mol), "Å²")
print("QED (parecido a fármaco, 0-1):", round(QED.qed(mol), 3))
print("Átomos totales (con H):", Chem.AddHs(mol).GetNumAtoms())

> 🔑 **Regla de los 5 de Lipinski:** un fármaco oral suele cumplir: masa ≤ 500 Da, LogP ≤ 5, ≤ 5 donadores de H y ≤ 10 aceptores de H. No es una ley, pero es un filtro muy usado en cribado virtual.

In [ ]:
from rdkit import Chem
writer = Chem.SDWriter("farmacos.sdf")
for nombre, smi in farmacos.items():
    m = Chem.MolFromSmiles(smi)
    m.SetProp("_Name", nombre)
    writer.write(m)
writer.close()
for m in Chem.SDMolSupplier("farmacos.sdf"):
    print(m.GetProp("_Name"))

*📁 Basado en: clase_9: rdkit_1_formatos.py, rdkit_2_API.py, rdkit_3_visualizar_moleculas.py y rdkit_5_Descriptors.py*

#### ✏️ Ejercicio: Filtro de Lipinski

Escribe `cumple_lipinski(smiles)` que devuelva `True` si la molécula cumple las 4 reglas de Lipinski, `False` si no, y `None` si el SMILES no es válido.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def cumple_lipinski(smiles):
    pass

Cuando termines, ejecuta la celda de comprobación (no la modifiques):

In [ ]:
# Comprobación automática
assert cumple_lipinski("CC(=O)OC1=CC=CC=C1C(=O)O") is True, "La aspirina cumple Lipinski"
assert cumple_lipinski("C1CC") is None, "Un SMILES inválido debe devolver None"
ciclosporina = "CCC1C(=O)N(CC(=O)N(C(C(=O)NC(C(=O)N(C(C(=O)NC(C(=O)NC(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N(C(C(=O)N1)C(C(C)CC=CC)O)C)C(C)C)C)CC(C)C)C)CC(C)C)C)C)C)CC(C)C)C)C(C)C)CC(C)C)C)C"
assert cumple_lipinski(ciclosporina) is False, "La ciclosporina (1202 Da) no cumple Lipinski"
print("✅ ¡Correcto!")

<details><summary>Ver solución explicada</summary>

Primero convierte y comprueba `None`. Después evalúa las cuatro condiciones con `Descriptors.MolWt`, `MolLogP`, `NumHDonors` y `NumHAcceptors`, combinadas con `and`.

```python
def cumple_lipinski(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return bool(Descriptors.MolWt(mol) <= 500 and Descriptors.MolLogP(mol) <= 5
                and Descriptors.NumHDonors(mol) <= 5 and Descriptors.NumHAcceptors(mol) <= 10)
```

</details>

#### 📌 Resumen del módulo
`Chem.MolFromSmiles / MolFromMolFile / MolFromInchi` crean objetos Mol (o None si fallan).`MolToSmiles` (canónico), `MolToInchi`, `MolToMolBlock` convierten formatos.`Draw.MolsToGridImage` dibuja varias moléculas.`Descriptors`: MolWt, MolLogP, TPSA, NumHDonors/Acceptors, NumRotatableBonds; `QED.qed`.
❓ ¿Qué representa el SMILES `c1ccccc1`?
    HexanoBencenoCiclohexano